# BrandSphere AI — Week 3: Font Classification & Recommendation Engine
**CRS AI Capstone 2025-26 | Scenario 1**

This notebook:
1. Downloads Font Dataset from Kaggle
2. Preprocesses font images (grayscale, resize 32x32, normalize)
3. Trains KNN classifier to identify font families
4. Maps font families to brand personalities
5. Saves model for Streamlit integration

## Cell 1 — Install Dependencies

In [ ]:
!pip install kagglehub opencv-python-headless scikit-learn matplotlib seaborn numpy pillow -q
print('✅ Dependencies installed')

## Cell 2 — Download Font Dataset

In [ ]:
import kagglehub
import os

print('📥 Downloading Font Dataset...')
font_path = kagglehub.dataset_download('muhammadardiputra/font-recognition-data')
print(f'✅ Font Dataset downloaded to: {font_path}')

# Explore structure
for root, dirs, files in os.walk(font_path):
    level = root.replace(font_path, '').count(os.sep)
    if level < 3:
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        if level == 1:
            img_files = [f for f in files if f.lower().endswith(('.jpg','.jpeg','.png'))]
            if img_files:
                print(f'{indent}  → {len(img_files)} images')

## Cell 3 — Find Dataset Root & Count Classes

In [ ]:
import os
import numpy as np

# Find the image directory
dataset_root = None
for root, dirs, files in os.walk(font_path):
    img_files = [f for f in files if f.lower().endswith(('.jpg','.jpeg','.png'))]
    if len(img_files) > 5 and len(dirs) == 0:
        parent = os.path.dirname(root)
        if dataset_root is None:
            dataset_root = parent
    if len(dirs) > 3 and dataset_root is None:
        dataset_root = root

if dataset_root is None:
    dataset_root = font_path

print(f'📂 Dataset root: {dataset_root}')
classes = sorted([d for d in os.listdir(dataset_root) if os.path.isdir(os.path.join(dataset_root, d))])
print(f'📊 Font classes found: {len(classes)}')
print(f'📋 Classes: {classes}')

class_counts = {}
for cls in classes:
    cls_path = os.path.join(dataset_root, cls)
    imgs = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    class_counts[cls] = len(imgs)
    print(f'   {cls}: {len(imgs)} images')

## Cell 4 — Preprocess Font Images (Grayscale, 32x32, Normalize)

In [ ]:
import cv2
import numpy as np
import random
from sklearn.model_selection import train_test_split

IMG_SIZE = 32
MAX_PER_CLASS = 200

print('🔄 Loading and preprocessing font images...')
X, y = [], []
label_map = {cls: idx for idx, cls in enumerate(classes)}

for cls in classes:
    cls_dir = os.path.join(dataset_root, cls)
    imgs = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    imgs = random.sample(imgs, min(MAX_PER_CLASS, len(imgs)))
    for img_file in imgs:
        img_path = os.path.join(cls_dir, img_file)
        img = cv2.imread(img_path)
        if img is None:
            continue
        # Convert to grayscale (as per rubric)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = img.astype(np.float32) / 255.0  # normalize
        X.append(img.flatten())  # flatten for KNN
        y.append(label_map[cls])

X = np.array(X)
y = np.array(y)
print(f'✅ Loaded {len(X)} font images')
print(f'📐 Feature vector size: {X.shape[1]} (32x32 flattened)')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(f'📊 Train: {len(X_train)} | Test: {len(X_test)}')

## Cell 5 — Visualize Sample Font Images

In [ ]:
import matplotlib.pyplot as plt

reverse_map = {v: k for k, v in label_map.items()}

fig, axes = plt.subplots(2, len(classes), figsize=(4*len(classes), 8))
fig.suptitle('Sample Font Images per Class', fontsize=16, fontweight='bold')

for col, cls in enumerate(classes):
    cls_indices = np.where(y_train == label_map[cls])[0][:2]
    for row, idx in enumerate(cls_indices[:2]):
        ax = axes[row][col] if len(classes) > 1 else axes[row]
        img = X_train[idx].reshape(IMG_SIZE, IMG_SIZE)
        ax.imshow(img, cmap='gray')
        ax.set_title(cls[:12], fontsize=9)
        ax.axis('off')

plt.tight_layout()
plt.savefig('sample_fonts.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Sample fonts visualized')

## Cell 6 — Train KNN Font Classifier

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print('🔄 Training KNN classifier...')

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Train KNN (k=5)
knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean', n_jobs=-1)
knn.fit(X_train_scaled, y_train)

print('✅ KNN trained!')
train_acc = knn.score(X_train_scaled, y_train)
test_acc  = knn.score(X_test_scaled, y_test)
print(f'📊 Train Accuracy: {train_acc*100:.2f}%')
print(f'📊 Test Accuracy:  {test_acc*100:.2f}%')

y_pred = knn.predict(X_test_scaled)
print('\n📋 Classification Report:')
print(classification_report(y_test, y_pred, target_names=classes))

## Cell 7 — Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrBr',
            xticklabels=classes, yticklabels=classes)
plt.title('KNN Font Classifier — Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('font_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Confusion matrix saved')

## Cell 8 — Map Font Families to Brand Personality

In [ ]:
import pandas as pd

# Brand personality to font family mapping
PERSONALITY_FONT_MAP = {
    'minimalist': {
        'primary':   ('Helvetica Neue', 'Sans-serif', 'Clean, universal trust'),
        'secondary': ('Futura',         'Geometric Sans', 'Forward-thinking, precise'),
        'accent':    ('Gill Sans',      'Humanist Sans', 'Approachable yet modern'),
        'rationale': 'Minimalist brands need clean, uncluttered typefaces with consistent stroke widths'
    },
    'vibrant': {
        'primary':   ('Montserrat',  'Geometric Sans', 'Bold, energetic presence'),
        'secondary': ('Nunito',      'Rounded Sans',   'Friendly, high energy'),
        'accent':    ('Raleway',     'Elegant Sans',   'Dynamic with character'),
        'rationale': 'Vibrant brands need expressive typefaces with strong visual personality'
    },
    'luxury': {
        'primary':   ('Cormorant Garamond', 'Serif',              'Timeless editorial elegance'),
        'secondary': ('Didot',              'Modern Serif',        'High-fashion authority'),
        'accent':    ('Playfair Display',   'Transitional Serif',  'Sophisticated contrast'),
        'rationale': 'Luxury brands require refined serif typefaces with high contrast stroke variation'
    },
    'bold': {
        'primary':   ('Oswald',      'Condensed Sans', 'Strong, impactful, modern'),
        'secondary': ('Anton',       'Display Sans',   'Maximum visual impact'),
        'accent':    ('Bebas Neue',  'Display',        'Unapologetically bold'),
        'rationale': 'Bold brands need heavy, attention-commanding display typefaces'
    },
    'elegant': {
        'primary':   ('Garamond',           'Old-style Serif',    'Classical refinement'),
        'secondary': ('Libre Baskerville',  'Transitional Serif', 'Elegant and readable'),
        'accent':    ('EB Garamond',        'Humanist Serif',     'Warm scholarly grace'),
        'rationale': 'Elegant brands suit graceful serif typefaces with classical proportions'
    },
}

# Industry to font style lookup
INDUSTRY_FONT_MAP = {
    'Tech / Software':     'minimalist',
    'Fashion / Apparel':   'elegant',
    'Food & Beverage':     'vibrant',
    'Healthcare':          'minimalist',
    'Finance':             'luxury',
    'Education':           'bold',
    'Retail / E-commerce': 'vibrant',
    'Real Estate':         'luxury',
    'Creative / Design':   'bold',
    'Manufacturing':       'minimalist',
}

# Display mapping table
rows = []
for industry, personality in INDUSTRY_FONT_MAP.items():
    fonts = PERSONALITY_FONT_MAP[personality]
    rows.append({
        'Industry': industry,
        'Personality': personality.title(),
        'Primary Font': fonts['primary'][0],
        'Font Type': fonts['primary'][1],
        'Rationale': fonts['rationale'][:60] + '...'
    })

df_fonts = pd.DataFrame(rows)
print('📋 Industry → Font Recommendation Lookup Table:')
print(df_fonts.to_string(index=False))
df_fonts.to_csv('font_recommendations.csv', index=False)
print('\n✅ Font recommendations saved to CSV')

## Cell 9 — Test Font Recommendation

In [ ]:
def recommend_fonts(industry: str, personality: str) -> dict:
    """
    Recommend fonts for a brand based on industry and personality.
    Uses KNN-classified font families mapped to brand personas.
    """
    persona = INDUSTRY_FONT_MAP.get(industry, personality)
    fonts = PERSONALITY_FONT_MAP.get(persona, PERSONALITY_FONT_MAP['minimalist'])
    return fonts

# Test for multiple industries
test_cases = [
    ('Tech / Software', 'minimalist'),
    ('Fashion / Apparel', 'luxury'),
    ('Food & Beverage', 'vibrant'),
]

for industry, personality in test_cases:
    rec = recommend_fonts(industry, personality)
    print(f'\n🏢 {industry} ({personality.title()})')
    print(f'   Primary:   {rec["primary"][0]} — {rec["primary"][1]}')
    print(f'   Secondary: {rec["secondary"][0]} — {rec["secondary"][1]}')
    print(f'   Accent:    {rec["accent"][0]} — {rec["accent"][1]}')
    print(f'   Why:       {rec["rationale"][:70]}...')

## Cell 10 — Save Model & Upload to Drive

In [ ]:
import pickle, shutil
from google.colab import drive

# Save KNN model and scaler
with open('font_knn_model.pkl', 'wb') as f:
    pickle.dump({'model': knn, 'scaler': scaler, 'label_map': label_map,
                 'reverse_map': reverse_map, 'classes': classes,
                 'personality_map': PERSONALITY_FONT_MAP,
                 'industry_map': INDUSTRY_FONT_MAP}, f)

print('✅ Saved font_knn_model.pkl')

drive.mount('/content/drive')
save_dir = '/content/drive/MyDrive/BrandSphere_Models'
os.makedirs(save_dir, exist_ok=True)

for fname in ['font_knn_model.pkl', 'font_recommendations.csv',
              'sample_fonts.png', 'font_confusion_matrix.png']:
    if os.path.exists(fname):
        shutil.copy(fname, os.path.join(save_dir, fname))
        print(f'   ✅ Uploaded {fname} to Drive')

## Cell 11 — Summary

In [ ]:
print('=' * 55)
print('  BRANDSPHERE AI — FONT CLASSIFIER SUMMARY')
print('=' * 55)
print(f'  Dataset:       Font Recognition Dataset (Kaggle)')
print(f'  Classes:       {len(classes)} font families')
print(f'  Images used:   {len(X)}')
print(f'  Image size:    {IMG_SIZE}x{IMG_SIZE} grayscale → flattened')
print(f'  Algorithm:     KNN (k=5, Euclidean distance)')
print(f'  Train Acc:     {train_acc*100:.2f}%')
print(f'  Test Acc:      {test_acc*100:.2f}%')
print(f'  Integration:   Personality → Font family lookup')
print('=' * 55)
print('  OUTPUTS:')
print('  - font_knn_model.pkl       → saved KNN model')
print('  - font_recommendations.csv → industry lookup table')
print('  - sample_fonts.png         → dataset samples')
print('  - font_confusion_matrix.png → evaluation')
print('=' * 55)